In [ ]:
# ============================================================================
# ADVANCED RESEARCH-GRADE EVALUATION SUITE
# ============================================================================
import os
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import models, backend as K
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             mean_absolute_error, mean_squared_error, r2_score,
                             classification_report, confusion_matrix)
from pathlib import Path
from google.colab import files

# --- 1. CONFIGURATION ---
print("🚀 Starting Advanced Evaluation Pipeline...")
OUTPUT_DIR = "advanced_evaluation_outputs"
if os.path.exists(OUTPUT_DIR): shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR)

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Paths & Config
DATASET_PATH = Path("/content/drive/MyDrive/Datasets/Nutrition5k/nutrition5k_dataset")
METADATA_PATH = DATASET_PATH / "metadata"
IMAGERY_PATH = DATASET_PATH / "imagery" / "realsense_overhead"
MODEL_PATH = "best_balanced_model_old.keras"
IMG_SIZE = 260
BATCH_SIZE = 32
NORM_FACTORS = np.array([1200.0, 100.0, 150.0, 100.0], dtype=np.float32)

# --- 2. DATA LOADING ---
def load_test_data():
    print("📂 Loading and cleaning dataset...")
    ignore_list = {
        'salt', 'pepper', 'olive oil', 'vegetable oil', 'sugar', 'brown sugar',
        'vinegar', 'soy sauce', 'lemon juice', 'orange juice', 'grapefruit juice',
        'mayonnaise', 'butter', 'ketchup', 'mustard', 'syrup', 'white wine', 'wine',
        'cream', 'sour cream', 'buttermilk', 'milk', 'flour', 'caesar dressing',
        'vinaigrette', 'salsa', 'pesto', 'plate only', 'deprecated', 'water',
        'lemon', 'lime', 'honey', 'maple syrup', 'balsamic vinegar', 'oil', 'spray', 'dressing'
    }
    data = []
    files = [METADATA_PATH / "dish_metadata_cafe1.csv", METADATA_PATH / "dish_metadata_cafe2.csv"]

    for file_path in files:
        if not file_path.exists(): continue
        try:
            df_raw = pd.read_csv(file_path, header=None, names=range(400), engine='python')
        except: continue

        for _, row in df_raw.iterrows():
            raw_nut = [row[1], row[3], row[4], row[5]]
            if any(pd.isna(x) for x in raw_nut): continue
            if float(raw_nut[0]) <= 1.0: continue

            ings = []
            for i in range(6, len(row), 7):
                if i+1 >= len(row): break
                raw_ing = row[i+1]
                if pd.notna(raw_ing):
                    clean = str(raw_ing).strip().lower()
                    if clean not in ignore_list and len(clean) > 2:
                        ings.append(clean)
            if not ings: continue

            img_path = IMAGERY_PATH / str(row[0]) / "rgb.png"
            if not img_path.exists(): continue

            nut_array = np.array([float(x) for x in raw_nut], dtype=np.float32)
            normalized_nut = np.clip(nut_array / NORM_FACTORS, 0.0, 1.0)
            data.append({'path': str(img_path), 'nutrition': normalized_nut, 'ingredients': ings})

    return pd.DataFrame(data)

df = load_test_data()
mlb = MultiLabelBinarizer()
binary_labels = mlb.fit_transform(df['ingredients'])
class_names = list(mlb.classes_)

# Recreate Test Split
_, test_df, _, test_y = train_test_split(df, binary_labels, test_size=0.15, random_state=42)
print(f"✅ Test Set Isolated: {len(test_df)} samples")

# --- 3. CREATE PIPELINE ---
def process_path(path, ingredients, nutrition):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return img, {'ing_out': ingredients, 'nut_out': nutrition}

def make_ds(dataframe, y_labels):
    paths = dataframe['path'].values
    nut_stack = np.vstack(dataframe['nutrition'].values).astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((paths, y_labels, nut_stack))
    ds = ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_ds = make_ds(test_df, test_y)

# --- 4. LOAD MODEL ---
def binary_focal_loss(gamma=2., alpha=.25):
    def focal_loss_fixed(y_true, y_pred): return 0
    return focal_loss_fixed
def keras_dummy_f1(y_true, y_pred): return 0

print(f"📥 Loading Model: {MODEL_PATH}...")
model = models.load_model(MODEL_PATH, custom_objects={'focal_loss_fixed': binary_focal_loss(), 'f1_score': keras_dummy_f1})

# --- 5. RUN FULL INFERENCE ---
print("⚡ Running Inference on Full Test Set...")
all_pred_ing, all_true_ing = [], []
all_pred_nut, all_true_nut = [], []

for images, targets in test_ds:
    preds = model.predict(images, verbose=0)
    all_pred_ing.append(preds[0])
    all_true_ing.append(targets['ing_out'].numpy())
    all_pred_nut.append(preds[1])
    all_true_nut.append(targets['nut_out'].numpy())

y_pred_ing = np.vstack(all_pred_ing)
y_true_ing = np.vstack(all_true_ing)
y_pred_nut = np.vstack(all_pred_nut)
y_true_nut = np.vstack(all_true_nut)

# --- 6. CALCULATE METRICS ---
print("📊 Generating Detailed Report...")
metrics_report = "=== ADVANCED MODEL EVALUATION REPORT ===\n\n"

# A. Ingredients
y_pred_ing_bin = (y_pred_ing > 0.3).astype(int)
metrics_report += "--- 1. INGREDIENT CLASSIFICATION ---\n"
metrics_report += f"Micro Precision: {precision_score(y_true_ing, y_pred_ing_bin, average='micro'):.4f}\n"
metrics_report += f"Micro Recall:    {recall_score(y_true_ing, y_pred_ing_bin, average='micro'):.4f}\n"
metrics_report += f"Micro F1 Score:  {f1_score(y_true_ing, y_pred_ing_bin, average='micro'):.4f}\n\n"

# Per-Class Report (Top 50 most common ingredients)
metrics_report += "--- PER-CLASS PERFORMANCE (Top 50 Ingredients) ---\n"
class_counts = np.sum(y_true_ing, axis=0)
sorted_indices = np.argsort(class_counts)[::-1][:50]
cls_report = classification_report(y_true_ing, y_pred_ing_bin, target_names=class_names, output_dict=True, zero_division=0)

metrics_report += f"{'Ingredient':<20} {'F1-Score':<10} {'Support':<10}\n"
metrics_report += "-"*45 + "\n"
top_50_names = []
top_50_f1 = []

for idx in sorted_indices:
    name = class_names[idx]
    score = cls_report[name]['f1-score']
    support = cls_report[name]['support']
    metrics_report += f"{name:<20} {score:<10.2f} {support:<10}\n"
    top_50_names.append(name)
    top_50_f1.append(score)
metrics_report += "\n"

# B. Nutrition
y_pred_real = y_pred_nut * NORM_FACTORS
y_true_real = y_true_nut * NORM_FACTORS
nutrients = ['Calories', 'Fat', 'Carbs', 'Protein']

metrics_report += "--- 2. NUTRITION REGRESSION ---\n"
metrics_report += f"{'Nutrient':<15} {'MAE':<10} {'RMSE':<10} {'R2':<10} {'Mean%Err':<10}\n"
metrics_report += "-"*60 + "\n"

for i, n in enumerate(nutrients):
    t = y_true_real[:, i]
    p = y_pred_real[:, i]
    mae = mean_absolute_error(t, p)
    rmse = np.sqrt(mean_squared_error(t, p))
    r2 = r2_score(t, p)
    # Mean Percentage Error (handling zeros)
    mpe = np.mean(np.abs((t - p) / (t + 1.0))) * 100
    metrics_report += f"{n:<15} {mae:<10.2f} {rmse:<10.2f} {r2:<10.4f} {mpe:<10.1f}%\n"

with open(f"{OUTPUT_DIR}/detailed_metrics.txt", "w") as f:
    f.write(metrics_report)

# --- 7. ADVANCED VISUALIZATIONS ---
print("🎨 Generating 6 Research Graphs...")
sns.set_style("whitegrid")

# Graph 1: Top 20 Ingredients F1 Score
plt.figure(figsize=(12, 6))
sns.barplot(x=top_50_f1[:20], y=top_50_names[:20], palette="viridis")
plt.title("Top 20 Ingredients Performance (F1 Score)")
plt.xlabel("F1 Score")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/graph_1_ingredient_performance.png")
plt.close()

# Graph 2: Nutrition Regression Scatter (4 Subplots)
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()
for i, n in enumerate(nutrients):
    t = y_true_real[:, i]
    p = y_pred_real[:, i]
    ax = axes[i]
    ax.scatter(t, p, alpha=0.3, s=15, color='royalblue')
    max_val = max(t.max(), p.max())
    ax.plot([0, max_val], [0, max_val], 'r--', lw=2)
    ax.set_title(f"{n}: Actual vs Predicted")
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
    r2 = r2_score(t, p)
    ax.text(0.05, 0.9, f"R² = {r2:.2f}", transform=ax.transAxes, fontsize=12, bbox=dict(facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/graph_2_nutrition_scatter.png")
plt.close()

# Graph 3: Error Distribution Histograms
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()
for i, n in enumerate(nutrients):
    errors = y_pred_real[:, i] - y_true_real[:, i]
    sns.histplot(errors, kde=True, ax=axes[i], color='purple', bins=50)
    axes[i].set_title(f"{n} Error Distribution")
    axes[i].set_xlabel("Error (Predicted - Actual)")
    axes[i].axvline(0, color='red', linestyle='--')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/graph_3_error_distribution.png")
plt.close()

# --- 8. SAVE 10 VISUAL SAMPLES ---
print("📸 Saving 10 Visual Samples...")
for batch_idx, (images, targets) in enumerate(test_ds.take(2)): # Take 2 batches to ensure we get 10 images
    b_preds = model.predict(images, verbose=0)

    start_idx = 0
    end_idx = min(len(images), 10 if batch_idx == 0 else 10 - BATCH_SIZE)
    if batch_idx == 0: end_idx = min(len(images), 10)
    elif batch_idx == 1:
        # Logic to continue saving if batch 1 was < 10 (unlikely with batch 32)
        break

    for i in range(end_idx):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

        # Image
        ax1.imshow(images[i].numpy().astype("uint8"))
        ax1.axis('off')
        ax1.set_title(f"Test Sample #{i+1}")

        # Text Info
        ax2.axis('off')
        true_n = targets['nut_out'][i].numpy() * NORM_FACTORS
        pred_n = b_preds[1][i] * NORM_FACTORS

        text = "NUTRITION ANALYSIS:\n" + "-"*35 + "\n"
        text += f"{'Item':<10} | {'Actual':>8} | {'Pred':>8} | {'Diff':>6}\n"
        text += "-"*35 + "\n"
        for j, m in enumerate(nutrients):
            diff = abs(true_n[j] - pred_n[j])
            text += f"{m:<10} | {true_n[j]:>8.0f} | {pred_n[j]:>8.0f} | {diff:>6.0f}\n"

        text += "\nINGREDIENTS:\n" + "-"*35 + "\n"
        t_idxs = np.where(targets['ing_out'][i].numpy() == 1)[0]
        p_idxs = np.where(b_preds[0][i] > 0.3)[0]
        t_names = [class_names[x] for x in t_idxs]
        p_names = [f"{class_names[x]} ({b_preds[0][i][x]:.2f})" for x in p_idxs]

        def wrap(lst): return "\n".join([", ".join(lst)[i:i+40] for i in range(0, len(", ".join(lst)), 40)])
        text += f"True:\n{wrap(t_names)}\n\nPred:\n{wrap(p_names)}"

        ax2.text(0, 1, text, fontsize=11, family='monospace', verticalalignment='top')
        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/sample_{batch_idx*BATCH_SIZE + i + 1}.png")
        plt.close()
    if batch_idx == 0: break # Just need 10 from first batch

# --- 9. ZIP & DOWNLOAD ---
print("📦 Zipping & Downloading...")
shutil.make_archive("advanced_evaluation_results", 'zip', OUTPUT_DIR)
files.download("advanced_evaluation_results.zip")
print("✅ Done! Check the text file in the zip for detailed per-class stats.")

🚀 Starting Advanced Evaluation Pipeline...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Loading and cleaning dataset...
✅ Test Set Isolated: 487 samples
📥 Loading Model: best_balanced_model_old.keras...
⚡ Running Inference on Full Test Set...
📊 Generating Detailed Report...
🎨 Generating 6 Research Graphs...


/tmp/ipython-input-4268852221.py:194: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_50_f1[:20], y=top_50_names[:20], palette="viridis")


📸 Saving 10 Visual Samples...
📦 Zipping & Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Done! Check the text file in the zip for detailed per-class stats.
